In [ ]:
!pip install admet_ai


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.0/17.0 MB 28.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.1/67.1 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.9 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of xarray to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.4/166.4 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 124.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 117.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━

In [ ]:
import torch
import numpy as np
from argparse import Namespace
from admet_ai import ADMETModel
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski, Crippen

torch.serialization.add_safe_globals([
    Namespace,
    np.core.multiarray._reconstruct,
    np.ndarray,
    np.dtype,
    np.dtypes.Float64DType
])

binding_scores = [0.4914195347628156, 0.8303721053466643, 1.0581474159307722, 0.7795295945044215, 0.3308276468591592, 0.9519032299358294, 1.1030335677176402, 0.7674287004159897]
ligand_smiles_list = ['CC1=C(C(=CC=C1)Cl)NC(=O)C2=CN=C(S2)NC3=CC(=NC(=N3)C)N4CCN(CC4)CCO', 'CC1=C(C=C(C=C1)C(=O)NC2=CC(=C(C=C2)CN3CCN(CC3)C)C(F)(F)F)C#CC4=CN=C5N4N=CC=C5', 'C1CN(C[C@@H]1O)C2=C(C=C(C=N2)C(=O)NC3=CC=C(C=C3)OC(F)(F)Cl)C4=CC=NN4', 'CNC(=O)C1=CC=CC=C1SC2=CC3=C(C=C2)C(=NN3)/C=C/C4=CC=CC=N4', 'CC1=C(C=C(C=C1)C(=O)NC2=CC(=CC(=C2)C(F)(F)F)N3C=C(N=C3)C)NC4=NC=CC(=N4)C5=CN=CC=C5', 'CC1=C(C=C(C=C1)NC(=O)C2=CC=C(C=C2)CN3CCN(CC3)C)NC4=NC=CC(=N4)C5=CN=CC=C5', 'CC1=C(C=C(C=C1)NC(=O)C2=CC(=C(C=C2)CN3CC[C@@H](C3)N(C)C)C(F)(F)F)NC4=NC=CC(=N4)C5=CN=CN=C5', 'CN1CCN(CC1)CCCOC2=C(C=C3C(=C2)N=CC(=C3NC4=CC(=C(C=C4Cl)Cl)OC)C#N)OC']

model = ADMETModel()
admet_df = model.predict(smiles=ligand_smiles_list)

admet_df = admet_df.reset_index().rename(columns={'index': 'SMILES'})

def lipinski_ok(mol):
    return (Descriptors.MolWt(mol) < 500 and
            Crippen.MolLogP(mol) < 5 and
            Lipinski.NumHDonors(mol) <= 5 and
            Lipinski.NumHAcceptors(mol) <= 10)

lipinski_flags = [lipinski_ok(Chem.MolFromSmiles(smi)) for smi in admet_df['SMILES']]
admet_df["lipinski_pass"] = lipinski_flags

good_admet = (
    (admet_df["HIA_Hou"] == True) &
    (admet_df["BBB_Martins"] == False) &
    (admet_df["hERG"] == False) &
    (admet_df["CYP3A4_Veith"] == False) &
    (admet_df["lipinski_pass"])
)
admet_df["admet_pass"] = good_admet

# --- Combine with binding scores and sort ---
rank_df = (
    admet_df
    .assign(binding_score=binding_scores)
    .sort_values(["admet_pass", "binding_score"], ascending=[False, False])
    .reset_index(drop=True)
)

print("--- DETAILED ADMET PROFILE ---")
print(rank_df[[
    "SMILES",
    "binding_score",
    "HIA_Hou",
    "BBB_Martins",
    "hERG",
    "CYP3A4_Veith",
    "lipinski_pass",
    "admet_pass" # The final combined result
]].head(10))

Loading pretrained parameter "encoder.encoder.0.cached_zero_vector".
Loading pretrained parameter "encoder.encoder.0.W_i.weight".
Loading pretrained parameter "encoder.encoder.0.W_h.weight".
Loading pretrained parameter "encoder.encoder.0.W_o.weight".
Loading pretrained parameter "encoder.encoder.0.W_o.bias".
Loading pretrained parameter "readout.1.weight".
Loading pretrained parameter "readout.1.bias".
Loading pretrained parameter "readout.4.weight".
Loading pretrained parameter "readout.4.bias".
Moving model to cuda
Loading pretrained parameter "encoder.encoder.0.cached_zero_vector".
Loading pretrained parameter "encoder.encoder.0.W_i.weight".
Loading pretrained parameter "encoder.encoder.0.W_h.weight".
Loading pretrained parameter "encoder.encoder.0.W_o.weight".
Loading pretrained parameter "encoder.encoder.0.W_o.bias".
Loading pretrained parameter "readout.1.weight".
Loading pretrained parameter "readout.1.bias".
Loading pretrained parameter "readout.4.weight".
Loading pretrained p

RDKit fingerprints: 100%|██████████| 8/8 [00:00<00:00, 18.99it/s]
/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
individual models:   0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00,  4.29it/s]

                                             
individual models:  20%|██        | 1/5 [00:00<00:01,  3.00it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00,  4.67it/s]

                                             
individual models:  40%|████      | 2/5 [00:00<00:00,  3.05it/s]

  0%|          | 0/1 [00:

--- DETAILED ADMET PROFILE ---
                                              SMILES  binding_score   HIA_Hou  \
0  CC1=C(C=C(C=C1)NC(=O)C2=CC(=C(C=C2)CN3CC[C@@H]...       1.103034  0.998798   
1  C1CN(C[C@@H]1O)C2=C(C=C(C=N2)C(=O)NC3=CC=C(C=C...       1.058147  0.998859   
2  CC1=C(C=C(C=C1)NC(=O)C2=CC=C(C=C2)CN3CCN(CC3)C...       0.951903  0.992016   
3  CC1=C(C=C(C=C1)C(=O)NC2=CC(=C(C=C2)CN3CCN(CC3)...       0.830372  0.998198   
4  CNC(=O)C1=CC=CC=C1SC2=CC3=C(C=C2)C(=NN3)/C=C/C...       0.779530  0.999184   
5  CN1CCN(CC1)CCCOC2=C(C=C3C(=C2)N=CC(=C3NC4=CC(=...       0.767429  0.985370   
6  CC1=C(C(=CC=C1)Cl)NC(=O)C2=CN=C(S2)NC3=CC(=NC(...       0.491420  0.995878   
7  CC1=C(C=C(C=C1)C(=O)NC2=CC(=CC(=C2)C(F)(F)F)N3...       0.330828  0.998695   

   BBB_Martins      hERG  CYP3A4_Veith  lipinski_pass  admet_pass  
0     0.786797  0.980993      0.978987          False       False  
1     0.285234  0.888585      0.780947           True       False  
2     0.597829  0.965251      0.857